# MobileADAS3D — MobileNetV4 Conv Small baseline

This notebook trains a fresh monocular-3D baseline on the canonical KITTI Chen 3,712/3,769 split. The current v2 experiment keeps the stride-16 FPN and MobileNetV4 Conv Small backbone, adds calibrated projected-center geometry, saves checkpoints to Google Drive, resumes after disconnects, and produces KITTI BEV/3D AP_R40 artifacts.

Before running: select **Runtime → Change runtime type → GPU** and make sure the repository changes containing this notebook are pushed to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import json, os, shlex, shutil, subprocess, sys, time

REPO_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
BRANCH = 'main'
PROJECT_DIR = Path('/content/mobile_adas3d')
EXPERIMENT_ID = 'mnv4_v2_calibrated_geometry_quality'
CONFIG = 'configs/kitti_mnv4_calibrated_geometry_v2.yaml'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
DRIVE_ARCHIVE_DIR = DRIVE_DATASET_ROOT / 'zips'
RUNTIME_DATASET_ROOT = Path('/content/kitti')
RUNTIME_ARCHIVE_ROOT = Path('/content/kitti_archive_stage')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_outputs/mnv4_conv_small_baseline')
STAGE_DATA_TO_LOCAL = True  # Faster epochs; source data remains in Drive.
PREFER_ARCHIVE_STAGE = True  # Much faster when Drive has KITTI zip/tar archives.
FORCE_RESTAGE_DATA = False  # Set True only when you want rsync to repair/re-copy local KITTI.
RUN_NAME = EXPERIMENT_ID
AUTO_RESUME = True
AUTO_RESUME_MATCH_RUN_NAME = True  # Prevents v2 from resuming v0/v1 checkpoints.
FORCE_NEW_RUN = False
REFERENCE_RUN_IDS = {
    'v0_earlystop_val_loss': '20260720_212816_baseline_mnv4_conv_small_stride16',
    'v1_long80_no_earlystop': '20260720_232755_mnv4_v1_long80_no_earlystop',
}

def run(command, cwd=None):
    print('+', ' '.join(shlex.quote(str(x)) for x in command))
    completed = subprocess.run([str(x) for x in command], cwd=cwd)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {command}')


In [ ]:
if not (PROJECT_DIR / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR])
else:
    run(['git', 'fetch', 'origin'], cwd=PROJECT_DIR)
    run(['git', 'checkout', BRANCH], cwd=PROJECT_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Repository:', PROJECT_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], cwd=PROJECT_DIR)
import torch, timm
print('torch:', torch.__version__)
print('timm:', timm.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))


## Stage KITTI onto the Colab runtime

Training directly from mounted Drive can bottleneck the data loader. This cell stages KITTI to temporary Colab storage for faster epochs while checkpoints and metrics remain in Drive. It first tries the fast archive path from `/content/drive/MyDrive/datasets/kitti/zips`, then falls back to folder `rsync` if no usable archive is found. It prints Drive path diagnostics, source/local counts, local disk space, progress, and writes `/content/kitti/.mobileadas3d_stage_manifest.json`. If Colab disconnects during staging, rerun this cell. Set `STAGE_DATA_TO_LOCAL=False` only if local disk space is insufficient.

In [ ]:
KITTI_SUBDIRS = {
    'training/image_2': {
        'suffix': '.png',
        'source_candidates': ['training/image_2', 'training/image_02'],
    },
    'training/label_2': {
        'suffix': '.txt',
        'source_candidates': ['training/label_2', 'training/label_02'],
    },
    'training/calib': {
        'suffix': '.txt',
        'source_candidates': ['training/calib'],
    },
}
KITTI_EXPECTED_COUNT = 7481
STAGE_MANIFEST = RUNTIME_DATASET_ROOT / '.mobileadas3d_stage_manifest.json'

def count_files(directory, suffix):
    if not directory.is_dir():
        return 0
    return sum(1 for path in directory.iterdir() if path.is_file() and path.suffix == suffix)

def collect_counts(root):
    return {subdir: count_files(root / subdir, spec['suffix']) for subdir, spec in KITTI_SUBDIRS.items()}

def print_counts(title, root, counts):
    print(f'\n{title}: {root}')
    for subdir in KITTI_SUBDIRS:
        print(f'  {subdir}: {counts[subdir]}')

def list_directory(path, limit=30):
    if not path.exists():
        print(f'  {path} does not exist')
        return
    if not path.is_dir():
        print(f'  {path} exists but is not a directory')
        return
    names = sorted(child.name for child in path.iterdir())[:limit]
    print(f'  {path}: {names}')

def is_archive(path):
    name = path.name.lower()
    return name.endswith(('.zip', '.tar', '.tar.gz', '.tgz'))

def find_archive_files(archive_dir):
    if not archive_dir.is_dir():
        return []
    return sorted(path for path in archive_dir.rglob('*') if path.is_file() and is_archive(path))

def resolve_source_subdirs(root):
    mapping = {}
    missing = []
    for destination_subdir, spec in KITTI_SUBDIRS.items():
        match = None
        for candidate in spec['source_candidates']:
            if (root / candidate).is_dir():
                match = candidate
                break
        if match is None:
            missing.append(destination_subdir)
        else:
            mapping[destination_subdir] = match
    return mapping, missing

def validate_kitti_source(root, label):
    source_mapping, missing = resolve_source_subdirs(root)
    if missing:
        print(f'\n{label} is missing required KITTI folders: {missing}')
        print('Accepted source folder names:')
        for destination_subdir, spec in KITTI_SUBDIRS.items():
            print(f"  {destination_subdir}: {spec['source_candidates']}")
        print('Nearby directories to inspect:')
        list_directory(root)
        list_directory(root / 'training')
        raise FileNotFoundError(f'{label} missing KITTI folders under {root}')
    if any(destination != source for destination, source in source_mapping.items()):
        print('\nUsing Drive source aliases and staging to canonical KITTI names:')
        for destination_subdir, source_subdir in source_mapping.items():
            if destination_subdir != source_subdir:
                print(f'  {source_subdir} -> {destination_subdir}')
    counts = {
        destination_subdir: count_files(root / source_subdir, KITTI_SUBDIRS[destination_subdir]['suffix'])
        for destination_subdir, source_subdir in source_mapping.items()
    }
    print_counts(f'{label} counts', root, counts)
    bad = {subdir: count for subdir, count in counts.items() if count != KITTI_EXPECTED_COUNT}
    if bad:
        raise RuntimeError(f'{label} expected {KITTI_EXPECTED_COUNT} files in each folder, got {bad}')
    return counts, source_mapping

def validate_kitti_staged_root(root, label):
    missing = [subdir for subdir in KITTI_SUBDIRS if not (root / subdir).is_dir()]
    if missing:
        print(f'\n{label} is missing canonical staged KITTI folders: {missing}')
        list_directory(root)
        list_directory(root / 'training')
        raise FileNotFoundError(f'{label} missing canonical KITTI folders under {root}')
    counts = collect_counts(root)
    print_counts(f'{label} counts', root, counts)
    bad = {subdir: count for subdir, count in counts.items() if count != KITTI_EXPECTED_COUNT}
    if bad:
        raise RuntimeError(f'{label} expected {KITTI_EXPECTED_COUNT} files in each folder, got {bad}')
    return counts

def local_disk_free_gb(path):
    usage_path = path if path.exists() else path.parent
    usage = shutil.disk_usage(usage_path)
    return usage.free / (1024 ** 3), usage.total / (1024 ** 3)

def write_stage_manifest(source_root, source_counts, staged_counts, source_mapping, complete):
    RUNTIME_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    payload = {
        'complete': complete,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'source': str(source_root),
        'destination': str(RUNTIME_DATASET_ROOT),
        'expected_count_per_folder': KITTI_EXPECTED_COUNT,
        'canonical_subdirs': list(KITTI_SUBDIRS),
        'source_candidates': {subdir: spec['source_candidates'] for subdir, spec in KITTI_SUBDIRS.items()},
        'source_mapping': source_mapping,
        'source_counts': source_counts,
        'staged_counts': staged_counts,
    }
    STAGE_MANIFEST.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
    print('Stage manifest:', STAGE_MANIFEST)

def stage_is_complete():
    if not STAGE_MANIFEST.is_file():
        return False
    try:
        manifest = json.loads(STAGE_MANIFEST.read_text())
    except json.JSONDecodeError:
        return False
    if manifest.get('complete') is not True:
        return False
    return all(count == KITTI_EXPECTED_COUNT for count in collect_counts(RUNTIME_DATASET_ROOT).values())

def copy_subdir_with_progress(source_root, destination_subdir, source_subdir, suffix):
    from tqdm.auto import tqdm

    source_dir = source_root / source_subdir
    destination_dir = RUNTIME_DATASET_ROOT / destination_subdir
    destination_dir.mkdir(parents=True, exist_ok=True)
    initial = min(count_files(destination_dir, suffix), KITTI_EXPECTED_COUNT)
    command = [
        'rsync',
        '-ah',
        '--partial',
        '--stats',
        f'{source_dir}/',
        f'{destination_dir}/',
    ]
    print(f'\nCopying: {source_subdir} -> {destination_subdir}')
    print('+', ' '.join(shlex.quote(str(x)) for x in command))
    process = subprocess.Popen(command)
    last = initial
    with tqdm(total=KITTI_EXPECTED_COUNT, initial=initial, desc=destination_subdir, unit='files', dynamic_ncols=True) as progress:
        while process.poll() is None:
            time.sleep(1.0)
            current = min(count_files(destination_dir, suffix), KITTI_EXPECTED_COUNT)
            progress.update(max(0, current - last))
            last = current
        current = min(count_files(destination_dir, suffix), KITTI_EXPECTED_COUNT)
        progress.update(max(0, current - last))
    if process.returncode != 0:
        raise RuntimeError(f'rsync failed for {source_subdir} with exit code {process.returncode}. Rerun this cell to resume, or inspect the rsync output above.')
    print(f'Finished {destination_subdir}: {count_files(destination_dir, suffix)}/{KITTI_EXPECTED_COUNT} files')

def copy_archive_to_local(archive_path):
    local_archive_dir = RUNTIME_ARCHIVE_ROOT / 'archives'
    local_archive_dir.mkdir(parents=True, exist_ok=True)
    local_archive = local_archive_dir / archive_path.name
    if local_archive.is_file() and local_archive.stat().st_size == archive_path.stat().st_size:
        print(f'Archive already local: {local_archive.name}')
        return local_archive
    command = ['rsync', '-ah', '--partial', '--progress', str(archive_path), str(local_archive)]
    print('\nCopying archive to local disk:')
    print('+', ' '.join(shlex.quote(str(x)) for x in command))
    run(command)
    return local_archive

def extract_archive(local_archive):
    extract_root = RUNTIME_ARCHIVE_ROOT / 'extracted'
    extract_root.mkdir(parents=True, exist_ok=True)
    name = local_archive.name.lower()
    print(f'\nExtracting archive: {local_archive.name}')
    if name.endswith('.zip'):
        run(['unzip', '-n', '-q', str(local_archive), '-d', str(extract_root)])
    elif name.endswith(('.tar', '.tar.gz', '.tgz')):
        run(['tar', '-xf', str(local_archive), '-C', str(extract_root)])
    else:
        raise RuntimeError(f'Unsupported archive type: {local_archive}')
    return extract_root

def candidate_kitti_roots(base, max_depth=3):
    candidates = [base]
    frontier = [(base, 0)]
    while frontier:
        current, depth = frontier.pop(0)
        if depth >= max_depth or not current.is_dir():
            continue
        for child in current.iterdir():
            if child.is_dir():
                candidates.append(child)
                frontier.append((child, depth + 1))
    return candidates

def find_extracted_kitti_source(extract_root):
    near_misses = []
    for candidate in candidate_kitti_roots(extract_root):
        source_mapping, missing = resolve_source_subdirs(candidate)
        if missing:
            continue
        counts = {
            destination_subdir: count_files(candidate / source_subdir, KITTI_SUBDIRS[destination_subdir]['suffix'])
            for destination_subdir, source_subdir in source_mapping.items()
        }
        if all(count == KITTI_EXPECTED_COUNT for count in counts.values()):
            print(f'\nFound complete KITTI source in extracted archives: {candidate}')
            print_counts('Extracted archive counts', candidate, counts)
            return candidate, counts, source_mapping
        near_misses.append((candidate, counts))
    if near_misses:
        print('\nArchives had KITTI-like folders, but counts were incomplete:')
        for candidate, counts in near_misses[:10]:
            print(f'  {candidate}: {counts}')
    return None, None, None

def try_stage_from_archives():
    archives = find_archive_files(DRIVE_ARCHIVE_DIR)
    print('\nArchive directory:', DRIVE_ARCHIVE_DIR)
    if not archives:
        print('No .zip/.tar archives found; falling back to folder rsync.')
        return None
    print('Found archives:')
    for archive in archives:
        print(f'  {archive.name} ({archive.stat().st_size / (1024 ** 3):.2f} GB)')
    extract_root = RUNTIME_ARCHIVE_ROOT / 'extracted'
    for archive in archives:
        local_archive = copy_archive_to_local(archive)
        extract_root = extract_archive(local_archive)
    archive_source_root, source_counts, source_mapping = find_extracted_kitti_source(extract_root)
    if archive_source_root is None:
        print('No complete KITTI training source found inside archives; falling back to folder rsync.')
        return None
    staged_counts = collect_counts(RUNTIME_DATASET_ROOT)
    write_stage_manifest(archive_source_root, source_counts, staged_counts, source_mapping, complete=False)
    for destination_subdir, source_subdir in source_mapping.items():
        copy_subdir_with_progress(archive_source_root, destination_subdir, source_subdir, KITTI_SUBDIRS[destination_subdir]['suffix'])
    staged_counts = validate_kitti_staged_root(RUNTIME_DATASET_ROOT, 'Local staged KITTI')
    write_stage_manifest(archive_source_root, source_counts, staged_counts, source_mapping, complete=True)
    return archive_source_root

print('Drive dataset root:', DRIVE_DATASET_ROOT)
print('Runtime dataset root:', RUNTIME_DATASET_ROOT)
print('rsync path:', shutil.which('rsync'))
free_gb, total_gb = local_disk_free_gb(RUNTIME_DATASET_ROOT)
print(f'Local disk free: {free_gb:.1f} GB / {total_gb:.1f} GB')
print('\nDrive directory check:')
list_directory(DRIVE_DATASET_ROOT.parent)
list_directory(DRIVE_DATASET_ROOT)
list_directory(DRIVE_ARCHIVE_DIR)

if STAGE_DATA_TO_LOCAL:
    if shutil.which('rsync') is None:
        raise RuntimeError('rsync is not available in this Colab runtime.')
    staged_counts = collect_counts(RUNTIME_DATASET_ROOT)
    print_counts('Current local staged counts', RUNTIME_DATASET_ROOT, staged_counts)
    if not FORCE_RESTAGE_DATA and stage_is_complete():
        print('\nLocal /content/kitti stage is already complete. Skipping copy.')
    else:
        archive_source_root = try_stage_from_archives() if PREFER_ARCHIVE_STAGE else None
        if archive_source_root is None:
            source_counts, source_mapping = validate_kitti_source(DRIVE_DATASET_ROOT, 'Drive KITTI source')
            write_stage_manifest(DRIVE_DATASET_ROOT, source_counts, staged_counts, source_mapping, complete=False)
            for destination_subdir, source_subdir in source_mapping.items():
                copy_subdir_with_progress(DRIVE_DATASET_ROOT, destination_subdir, source_subdir, KITTI_SUBDIRS[destination_subdir]['suffix'])
            staged_counts = validate_kitti_staged_root(RUNTIME_DATASET_ROOT, 'Local staged KITTI')
            write_stage_manifest(DRIVE_DATASET_ROOT, source_counts, staged_counts, source_mapping, complete=True)
    DATASET_ROOT = RUNTIME_DATASET_ROOT
else:
    source_counts, source_mapping = validate_kitti_source(DRIVE_DATASET_ROOT, 'Drive KITTI source')
    if any(destination != source for destination, source in source_mapping.items()):
        raise RuntimeError('STAGE_DATA_TO_LOCAL=False cannot train directly from alias folders like image_02/label_02. Set STAGE_DATA_TO_LOCAL=True so the notebook stages canonical image_2/label_2 folders.')
    DATASET_ROOT = DRIVE_DATASET_ROOT
print('\nTraining dataset root:', DATASET_ROOT)


## Canonical split and full preflight

This fails before training unless all 7,481 images, labels, and calibration files exist; the split is exactly 3,712/3,769 with no overlap; CUDA works; pretrained MobileNetV4 loads; output shapes remain stride 16; and a real KITTI loss is finite.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run([sys.executable, 'scripts/prepare_kitti_chen_split.py', '--config', CONFIG, '--profile', 'colab_drive', '--output-dir', SPLIT_DIR], cwd=PROJECT_DIR)
COMMON_NO_RUN_NAME = ['--config', CONFIG, '--profile', 'colab_drive', '--dataset-root', DATASET_ROOT, '--split-dir', SPLIT_DIR, '--output-dir', OUTPUT_DIR]
COMMON = ['--config', CONFIG, '--profile', 'colab_drive', '--run-name', RUN_NAME, '--dataset-root', DATASET_ROOT, '--split-dir', SPLIT_DIR, '--output-dir', OUTPUT_DIR]
run([sys.executable, 'scripts/check_kitti_splits.py', *COMMON_NO_RUN_NAME], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/check_training_ready.py', *COMMON_NO_RUN_NAME, '--require-cuda', '--report', OUTPUT_DIR / 'training_preflight.json'], cwd=PROJECT_DIR)


## Train or resume

`latest.pt` is atomically replaced after every epoch. With `AUTO_RESUME=True`, rerunning this cell after a Colab disconnect continues the newest run in this dedicated output directory. This cell streams training output live, saves the same log under `colab_logs/`, and prints the last log lines automatically on failure. Epoch snapshots are retained every 10 epochs.

In [ ]:
def print_tail(path, lines=80):
    if not path.is_file():
        print(f'No log file found: {path}')
        return
    text_lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
    print(f'\nLast {min(lines, len(text_lines))} log lines from {path}:')
    for line in text_lines[-lines:]:
        print(line)

def run_streamed(command, cwd, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    command = [str(x) for x in command]
    print('+', ' '.join(shlex.quote(x) for x in command))
    print('Live log:', log_path)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    with log_path.open('w', encoding='utf-8') as log_file:
        log_file.write('+ ' + ' '.join(shlex.quote(x) for x in command) + '\n')
        log_file.flush()
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code != 0:
        print_tail(log_path)
        raise RuntimeError(f'Training command failed with exit code {return_code}. Full log: {log_path}')

def describe_checkpoint(path):
    if path is None:
        return
    print('Resume checkpoint:', path)
    print(f'  exists: {path.is_file()}')
    if path.is_file():
        print(f'  size: {path.stat().st_size / (1024 ** 2):.1f} MB')
        print(f'  modified: {datetime.fromtimestamp(path.stat().st_mtime).isoformat()}')

def load_training_epochs(config_path):
    import yaml

    config_data = yaml.safe_load((PROJECT_DIR / config_path).read_text())
    return int(config_data['training']['epochs'])

def load_checkpoint_summary(path):
    if path is None or not path.is_file():
        return None
    import torch

    checkpoint = torch.load(path, map_location='cpu')
    return {
        'epoch': int(checkpoint.get('epoch', 0)),
        'global_step': int(checkpoint.get('global_step', 0)),
        'best_metric': checkpoint.get('best_metric', checkpoint.get('metric_value')),
    }

print('Experiment ID:', EXPERIMENT_ID)
print('Config:', CONFIG)
print('Run name:', RUN_NAME)
print('Reference runs:', REFERENCE_RUN_IDS)
MAX_EPOCHS = load_training_epochs(CONFIG)
print('Configured max epochs:', MAX_EPOCHS)
print('Training dataset root:', DATASET_ROOT)
print_counts('Training dataset counts', DATASET_ROOT, collect_counts(DATASET_ROOT))
print('Output directory:', OUTPUT_DIR)
print('Runs directory:', OUTPUT_DIR / 'runs')
all_latest_candidates = sorted((OUTPUT_DIR / 'runs').glob('*/checkpoints/latest.pt'), key=lambda p: p.stat().st_mtime)
if AUTO_RESUME_MATCH_RUN_NAME:
    latest_candidates = [path for path in all_latest_candidates if RUN_NAME in path.parent.parent.name]
else:
    latest_candidates = all_latest_candidates
if FORCE_NEW_RUN:
    latest_candidates = []
print(f'All latest checkpoint candidates: {len(all_latest_candidates)}')
print(f'Matched resume candidates for {RUN_NAME}: {len(latest_candidates)}')
for candidate in latest_candidates[-5:]:
    print(f'  {candidate} ({candidate.stat().st_size / (1024 ** 2):.1f} MB)')

resume_checkpoint = latest_candidates[-1] if (AUTO_RESUME and latest_candidates and not FORCE_NEW_RUN) else None
train_command = [sys.executable, '-u', 'scripts/train_mobile_adas3d.py', *COMMON]
if resume_checkpoint is not None:
    train_command += ['--resume', resume_checkpoint]
    TRAIN_RUN_DIR = resume_checkpoint.parent.parent
    print('Resuming existing run.')
    describe_checkpoint(resume_checkpoint)
    checkpoint_summary = load_checkpoint_summary(resume_checkpoint)
    print('  saved epoch:', checkpoint_summary['epoch'])
    print('  saved global_step:', checkpoint_summary['global_step'])
    print('  saved best_metric:', checkpoint_summary['best_metric'])
else:
    print('Starting a new experiment run.')
    checkpoint_summary = None

LOG_DIR = OUTPUT_DIR / 'colab_logs'
LOG_PATH = LOG_DIR / f"train_{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
checkpoint_is_complete = checkpoint_summary is not None and checkpoint_summary['epoch'] >= MAX_EPOCHS
if checkpoint_is_complete:
    print(f'\nSelected checkpoint already reached epoch {checkpoint_summary["epoch"]}/{MAX_EPOCHS}.')
    print('No training loop will run for this completed experiment.')
    print('To launch a new comparable run, set FORCE_NEW_RUN=True or change EXPERIMENT_ID/RUN_NAME before running this cell.')
else:
    run_streamed(train_command, cwd=PROJECT_DIR, log_path=LOG_PATH)

if resume_checkpoint is None:
    run_dirs = list((OUTPUT_DIR / 'runs').glob('*'))
    if not run_dirs:
        if not checkpoint_is_complete:
            print_tail(LOG_PATH)
        raise FileNotFoundError(f'No run directories found under {OUTPUT_DIR / "runs"}')
    TRAIN_RUN_DIR = max(run_dirs, key=lambda p: p.stat().st_mtime)

BEST_CHECKPOINT = TRAIN_RUN_DIR / 'checkpoints' / 'best.pt'
LATEST_CHECKPOINT = TRAIN_RUN_DIR / 'checkpoints' / 'latest.pt'
if not BEST_CHECKPOINT.is_file():
    if not checkpoint_is_complete:
        print_tail(LOG_PATH)
    raise FileNotFoundError(f'Best checkpoint missing: {BEST_CHECKPOINT}')
print('\nTraining finished or checkpoint already complete.')
print('Run directory:', TRAIN_RUN_DIR)
print('Latest checkpoint:', LATEST_CHECKPOINT)
print('Best checkpoint:', BEST_CHECKPOINT)
if not checkpoint_is_complete:
    print('Training log:', LOG_PATH)


## Evaluate the latest checkpoint on KITTI AP_R40

For v2, `latest.pt` can outperform `best.pt` because `best.pt` is still selected by validation loss, not AP. This evaluates all 3,769 validation images at a low score floor and writes `kitti_r40_metrics.csv`, `kitti_r40_summary.json`, and raw KITTI-format predictions. Only results with `complete_split: true` are reportable.

In [ ]:
EVAL_CHECKPOINT = LATEST_CHECKPOINT
EVAL_DIR = TRAIN_RUN_DIR / 'kitti_r40_latest'
run([sys.executable, 'scripts/evaluate_kitti_r40.py', '--config', CONFIG, '--profile', 'colab_drive', '--dataset-root', DATASET_ROOT, '--split-dir', SPLIT_DIR, '--checkpoint', EVAL_CHECKPOINT, '--split', 'val', '--score-threshold', '0.001', '--topk', '300', '--nms-iou-threshold', '0.5', '--output-dir', EVAL_DIR], cwd=PROJECT_DIR)

import json, pandas as pd
summary = json.loads((EVAL_DIR / 'kitti_r40_summary.json').read_text())
assert summary['complete_split'] and summary['evaluated_images'] == 3769
metrics = pd.DataFrame(summary['metrics'])
display(metrics.pivot_table(index=['metric', 'class_name'], columns='difficulty', values='ap_r40').round(3))
print('Evaluation checkpoint:', EVAL_CHECKPOINT)
print('Evaluation artifacts:', EVAL_DIR)


## Sweep AP across saved checkpoints

Run this after training to compare `epoch_*.pt`, `best.pt`, and `latest.pt` by KITTI AP_R40. The sweep reuses completed per-checkpoint summaries if Colab disconnects, skips per-frame prediction txt files by default to avoid Drive clutter, and writes `checkpoint_ap_summary.csv`, `checkpoint_ap_metrics_long.csv`, and `checkpoint_ap_summary.json`.

In [ ]:
SWEEP_DIR = TRAIN_RUN_DIR / 'checkpoint_ap_sweep_val'
run([sys.executable, 'scripts/sweep_kitti_r40_checkpoints.py', '--config', CONFIG, '--profile', 'colab_drive', '--dataset-root', DATASET_ROOT, '--split-dir', SPLIT_DIR, '--run-dir', TRAIN_RUN_DIR, '--split', 'val', '--score-threshold', '0.001', '--topk', '300', '--nms-iou-threshold', '0.5', '--output-dir', SWEEP_DIR], cwd=PROJECT_DIR)

import pandas as pd
sweep = pd.read_csv(SWEEP_DIR / 'checkpoint_ap_summary.csv')
columns = ['checkpoint', 'epoch', 'ap_3d_Car_moderate', 'mean_3d_moderate', 'ap_bev_Car_moderate', 'mean_bev_moderate']
display(sweep.sort_values('ap_3d_Car_moderate', ascending=False).head(10)[columns].round(3))
print('Sweep artifacts:', SWEEP_DIR)


## Optional TensorBoard

Run the following cell while training or after it finishes.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/mobile_adas3d_outputs/mnv4_conv_small_baseline/runs
